In [3]:

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pyart

BASE = Path("/Volumes/My Passport/hd2_trash_to_check")

FOLDERS = [
    "20211024.2",
    "20211024.3",
    "20211025.2",
    "20211025.3",
    "20211026.2",
    "20211026.3",
]

for folder_name in FOLDERS:
    folder = BASE / folder_name

    print("\n" + "=" * 90)
    print(f"FOLDER: {folder_name}")
    print("=" * 90)

    if not folder.is_dir():
        print("Folder not found")
        continue

    files = sorted(folder.rglob("*.mdv"))

    if not files:
        print("No .mdv files found")
        continue

    path = files[0]
    print(f"FILE: {path.name}")
    print(f"SIZE: {path.stat().st_size / 1024**2:.2f} MB")

    try:
        radar = pyart.io.read(str(path))

        print("\nRADAR INFORMATION")
        print("Scan type:", radar.scan_type)
        print("Number of rays:", radar.nrays)
        print("Number of gates:", radar.ngates)
        print("Number of sweeps:", radar.nsweeps)

        print("\nTIMESTAMPS")
        print("Time units:", radar.time["units"])

        times = pyart.util.datetimes_from_radar(radar)

        if len(times):
            print("First ray:", times[0])
            print("Last ray: ", times[-1])
            print("Duration: ", times[-1] - times[0])

        print("\nAVAILABLE FIELDS")
        for name, field in radar.fields.items():
            data = field["data"]
            print(f"\n  {name}")
            print(f"    shape: {data.shape}")
            print(f"    units: {field.get('units', 'N/A')}")
            print(f"    long_name: {field.get('long_name', 'N/A')}")
            print(f"    dtype: {data.dtype}")

        print("\nSWEEP INFORMATION")
        print("Fixed angles:", radar.fixed_angle["data"])
        print("Sweep modes:", radar.sweep_mode["data"])

        print("\nRADAR LOCATION")
        print("Latitude:", radar.latitude["data"])
        print("Longitude:", radar.longitude["data"])
        print("Altitude:", radar.altitude["data"])

        print("\nMETADATA")
        for key, value in radar.metadata.items():
            print(f"  {key}: {str(value)[:200]}")

    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")



## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly supported
## by the U.S. Department of Energy Office of Science as part of
## the Atmospheric Radiation Measurement (ARM) User Facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119


FOLDER: 20211024.2
FILE: 000003.mdv
SIZE: 0.03 MB
ERROR: TypeError: Unknown or unsupported file format: UNKNOWN

FOLDER: 20211024.3
FILE: 000003.mdv
SIZE: 0.06 MB
ERROR: TypeError: Unknown or unsupported file format: UNKNOWN

FOLDER: 20211025.2
FILE: 000026.mdv
SIZE: 0.01 MB
ERROR: TypeError: Unknown or unsupported file format: UNKNOWN

FOLDER: 20211025.3
FILE: 000026.mdv
SIZE: 0.01 MB
ERROR: TypeError: Unknown or unsupported file format: UNKNOWN

FOLDER: 20211026.2
FILE: 000022.mdv
SIZE: 0.08 MB
ERROR: TypeError: Unknown or unsupported file format: UNKNOWN

FOLDER: 20211026.3
FI

In [4]:

from pathlib import Path
import struct
import gzip

BASE = Path("/Volumes/My Passport/hd2_trash_to_check")

FOLDERS = [
    "20211024.2",
    "20211024.3",
    "20211025.2",
    "20211025.3",
    "20211026.2",
    "20211026.3",
]

for folder_name in FOLDERS:
    folder = BASE / folder_name

    print("\n" + "=" * 80)
    print("FOLDER:", folder_name)
    print("=" * 80)

    files = sorted(folder.rglob("*.mdv"))

    if not files:
        print("No MDV files found")
        continue

    path = files[0]

    print("File:", path.name)
    print("Size:", round(path.stat().st_size / 1024**2, 2), "MB")

    with open(path, "rb") as f:
        header = f.read(256)

    print("\nFirst 64 bytes (hex):")
    print(header[:64].hex(" "))

    print("\nFirst 64 bytes (ASCII):")
    print("".join(
        chr(b) if 32 <= b <= 126 else "."
        for b in header[:64]
    ))

    print("\nFormat checks:")

    if header.startswith(b"\x89HDF\r\n\x1a\n"):
        print("HDF5 detected")
    elif header.startswith(b"\x1f\x8b"):
        print("GZIP detected")
    elif header.startswith(b"CDF"):
        print("NetCDF classic detected")
    else:
        print("No standard HDF5/GZIP/NetCDF signature")

    # MDV commonly uses big-endian Fortran-style record markers.
    if len(header) >= 8:
        first_int = struct.unpack(">i", header[:4])[0]
        second_int = struct.unpack(">i", header[4:8])[0]

        print("\nFirst two big-endian integers:")
        print("  Integer 1:", first_int)
        print("  Integer 2:", second_int)

        if first_int == 1016:
            print("  Possible MDV master header (1016-byte record)")



FOLDER: 20211024.2
File: 000003.mdv
Size: 0.03 MB

First 64 bytes (hex):
00 00 03 f8 00 00 37 3e 00 00 00 02 61 74 a2 03 00 00 00 00 61 74 a0 e7 61 74 a2 03 61 74 a2 03 61 74 a4 3b 00 00 00 01 00 00 00 db 00 00 00 03 00 00 00 00 00 00 00 00 00 00 00 09 00 00 00 12

First 64 bytes (ASCII):
......7>....at......at..at..at..at.;............................

Format checks:
No standard HDF5/GZIP/NetCDF signature

First two big-endian integers:
  Integer 1: 1016
  Integer 2: 14142
  Possible MDV master header (1016-byte record)

FOLDER: 20211024.3
File: 000003.mdv
Size: 0.06 MB

First 64 bytes (hex):
00 00 03 f8 00 00 37 3e 00 00 00 02 61 74 a2 03 00 00 00 00 61 74 a0 e7 61 74 a2 03 61 74 a2 03 61 74 a4 3b 00 00 00 01 00 00 00 db 00 00 00 03 00 00 00 00 00 00 00 00 00 00 00 09 00 00 00 04

First 64 bytes (ASCII):
......7>....at......at..at..at..at.;............................

Format checks:
No standard HDF5/GZIP/NetCDF signature

First two big-endian integers:
  Integer 1: 1016
  Integer 2

In [5]:

from pathlib import Path
from datetime import datetime, timezone
import struct

BASE = Path("/Volumes/My Passport/hd2_trash_to_check")

FOLDERS = [
    "20211024.2",
    "20211024.3",
    "20211025.2",
    "20211025.3",
    "20211026.2",
    "20211026.3",
]

def int32(buf, offset):
    return struct.unpack_from(">i", buf, offset)[0]

def timestamp(value):
    if value <= 0:
        return str(value)
    try:
        return datetime.fromtimestamp(
            value, tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")
    except (ValueError, OverflowError, OSError):
        return str(value)

def cstring(buf):
    return buf.split(b"\x00", 1)[0].decode(
        "utf-8", errors="replace"
    ).strip()

def inspect_mdv(path):
    print(f"\nFILE: {path.name}")
    print(f"SIZE: {path.stat().st_size / 1024**2:.3f} MB")

    with open(path, "rb") as f:
        header = f.read(1024)

        if len(header) < 1024:
            print("Incomplete MDV master header")
            return

        record_size = int32(header, 0)
        struct_id = int32(header, 4)

        if record_size != 1016 or struct_id != 14142:
            print("Unexpected MDV header structure")
            return

        print("\nMASTER HEADER")
        print("Format revision:", int32(header, 8))

        times = {
            "Generation": int32(header, 12),
            "Begin": int32(header, 20),
            "End": int32(header, 24),
            "Centroid": int32(header, 28),
            "Expire": int32(header, 32),
        }

        print("\nTIMESTAMPS")
        for name, value in times.items():
            print(f"  {name:12s}: {timestamp(value)}")

        # Standard MDV master-header integer offsets.
        n_fields = int32(header, 96)
        n_vlevels = int32(header, 100)
        n_chunks = int32(header, 104)

        field_hdr_offset = int32(header, 108)

        print("\nSTRUCTURE")
        print("Fields:", n_fields)
        print("Vertical-level headers:", n_vlevels)
        print("Chunks:", n_chunks)
        print("Field-header offset:", field_hdr_offset)

        # Sanity check before reading field headers.
        if not (0 <= n_fields <= 1000):
            print("Field count is implausible; header layout needs checking.")
            return

        if not (0 <= field_hdr_offset < path.stat().st_size):
            print("Field-header offset is invalid; header layout needs checking.")
            return

        print("\nFIELD HEADERS")

        # MDV field headers are 416 bytes of content,
        # wrapped in 4-byte Fortran record markers.
        FIELD_RECORD_SIZE = 424

        for i in range(n_fields):
            f.seek(field_hdr_offset + i * FIELD_RECORD_SIZE)
            raw = f.read(FIELD_RECORD_SIZE)

            if len(raw) < FIELD_RECORD_SIZE:
                print(f"  Field {i}: incomplete header")
                break

            print(f"\n  FIELD {i + 1}")
            print("  Header marker:", int32(raw, 0))
            print("  Structure ID:", int32(raw, 4))

            # Show string candidates in the field header.
            # This is exploratory: offsets are not yet assumed.
            for start in range(0, len(raw) - 16, 4):
                segment = raw[start:start + 32]
                value = cstring(segment)

                if (
                    len(value) >= 3
                    and all(32 <= ord(c) < 127 for c in value)
                    and any(c.isalpha() for c in value)
                ):
                    print(f"    text at offset {start}: {value!r}")

for folder_name in FOLDERS:
    folder = BASE / folder_name

    print("\n" + "=" * 90)
    print("FOLDER:", folder_name)
    print("=" * 90)

    files = sorted(folder.rglob("*.mdv"))

    if not files:
        print("No MDV files found")
        continue

    try:
        inspect_mdv(files[0])
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")



FOLDER: 20211024.2

FILE: 000003.mdv
SIZE: 0.030 MB

MASTER HEADER
Format revision: 2

TIMESTAMPS
  Generation  : 2021-10-24 00:00:03 UTC
  Begin       : 2021-10-23 23:55:19 UTC
  End         : 2021-10-24 00:00:03 UTC
  Centroid    : 2021-10-24 00:00:03 UTC
  Expire      : 2021-10-24 00:09:31 UTC

STRUCTURE
Fields: 1024
Vertical-level headers: 2272
Chunks: 5344
Field-header offset: 0
Field count is implausible; header layout needs checking.

FOLDER: 20211024.3

FILE: 000003.mdv
SIZE: 0.060 MB

MASTER HEADER
Format revision: 2

TIMESTAMPS
  Generation  : 2021-10-24 00:00:03 UTC
  Begin       : 2021-10-23 23:55:19 UTC
  End         : 2021-10-24 00:00:03 UTC
  Centroid    : 2021-10-24 00:00:03 UTC
  Expire      : 2021-10-24 00:09:31 UTC

STRUCTURE
Fields: 1024
Vertical-level headers: 2272
Chunks: 5344
Field-header offset: 0
Field count is implausible; header layout needs checking.

FOLDER: 20211025.2

FILE: 000026.mdv
SIZE: 0.014 MB

MASTER HEADER
Format revision: 2

TIMESTAMPS
  Generat

In [6]:

from pathlib import Path
import struct

BASE = Path("/Volumes/My Passport/hd2_trash_to_check")

FOLDERS = [
    "20211024.2",
    "20211024.3",
    "20211025.2",
    "20211025.3",
    "20211026.2",
    "20211026.3",
]

def be_int(buf, offset):
    return struct.unpack_from(">i", buf, offset)[0]

for folder_name in FOLDERS:
    folder = BASE / folder_name
    files = sorted(folder.rglob("*.mdv"))

    print("\n" + "=" * 80)
    print("FOLDER:", folder_name)
    print("=" * 80)

    if not files:
        print("No MDV files")
        continue

    path = files[0]
    file_size = path.stat().st_size

    with open(path, "rb") as f:
        header = f.read(1024)

        print("File:", path.name)
        print("File size:", file_size, "bytes")

        print("\nMASTER HEADER INTEGERS")
        print(f"{'Offset':>8} {'Value':>12} {'Hex':>12}")
        print("-" * 36)

        for offset in range(36, 180, 4):
            value = be_int(header, offset)
            print(f"{offset:8d} {value:12d} {value & 0xffffffff:08x}")

        print("\nPOSSIBLE FIELD HEADER LOCATIONS")

        # Search for MDV field-header structure identifier.
        # Standard MDV field header ID is 14143.
        f.seek(0)
        data = f.read(min(file_size, 100000))

        signature = struct.pack(">i", 14143)

        positions = []
        start = 0

        while True:
            pos = data.find(signature, start)

            if pos < 0:
                break

            positions.append(pos)
            start = pos + 4

        if positions:
            for pos in positions[:20]:
                print(
                    f"  Field header ID at byte {pos}; "
                    f"possible record start at {pos - 4}"
                )
        else:
            print("  No standard field-header IDs found")



FOLDER: 20211024.2
File: 000003.mdv
File size: 31226 bytes

MASTER HEADER INTEGERS
  Offset        Value          Hex
------------------------------------
      36            1 00000001
      40          219 000000db
      44            3 00000003
      48            0 00000000
      52            0 00000000
      56            9 00000009
      60           18 00000012
      64            1 00000001
      68            1 00000001
      72            0 00000000
      76            3 00000003
      80          800 00000320
      84          800 00000320
      88            1 00000001
      92            3 00000003
      96         1024 00000400
     100         2272 000008e0
     104         5344 000014e0
     108            0 00000000
     112            0 00000000
     116            0 00000000
     120            0 00000000
     124            0 00000000
     128            0 00000000
     132            0 00000000
     136            0 00000000
     140            0 00000000
     14

In [7]:

from pathlib import Path
import struct
import re

BASE = Path("/Volumes/My Passport/hd2_trash_to_check")

FOLDERS = [
    "20211024.2",
    "20211024.3",
    "20211025.2",
    "20211025.3",
    "20211026.2",
    "20211026.3",
]

def be_i32(buf, offset):
    return struct.unpack_from(">i", buf, offset)[0]

def extract_strings(buf, min_length=2):
    """Find printable strings in a binary header."""
    matches = re.finditer(
        rb"[\x20-\x7e]{" + str(min_length).encode() + rb",}",
        buf
    )
    return [
        (m.start(), m.group().decode("ascii"))
        for m in matches
    ]

for folder_name in FOLDERS:
    folder = BASE / folder_name
    files = sorted(folder.rglob("*.mdv"))

    print("\n" + "=" * 85)
    print("FOLDER:", folder_name)
    print("=" * 85)

    if not files:
        print("No MDV files found")
        continue

    path = files[0]

    with open(path, "rb") as f:
        master = f.read(1024)

        n_fields = be_i32(master, 44)
        field_offset = be_i32(master, 96)

        print("File:", path.name)
        print("Number of fields:", n_fields)

        for i in range(n_fields):
            f.seek(field_offset + i * 416)
            raw = f.read(416)

            print(f"\n{'-' * 60}")
            print(f"FIELD {i + 1}")
            print("-" * 60)

            print("Header size:", be_i32(raw, 0))
            print("Structure ID:", be_i32(raw, 4))

            print("\nTEXT STRINGS:")
            for offset, text in extract_strings(raw, 3):
                print(f"  offset {offset:3d}: {text[:120]}")

            print("\nFIRST 30 INTEGER VALUES:")
            for offset in range(8, 128, 4):
                print(
                    f"  offset {offset:3d}: "
                    f"{be_i32(raw, offset)}"
                )



FOLDER: 20211024.2
File: 000003.mdv
Number of fields: 3

------------------------------------------------------------
FIELD 1
------------------------------------------------------------
Header size: 408
Structure ID: 14143

TEXT STRINGS:
  offset 275: PBI
  offset 284: Tops10
  offset 348: Tops10

FIRST 30 INTEGER VALUES:
  offset   8: 0
  offset  12: 0
  offset  16: 0
  offset  20: 0
  offset  24: 0
  offset  28: 0
  offset  32: 0
  offset  36: 800
  offset  40: 800
  offset  44: 1
  offset  48: 8
  offset  52: 5
  offset  56: 4
  offset  60: 6884
  offset  64: 9746
  offset  68: 0
  offset  72: 0
  offset  76: 0
  offset  80: 0
  offset  84: 0
  offset  88: 0
  offset  92: 0
  offset  96: 0
  offset 100: 0
  offset 104: 0
  offset 108: 5
  offset 112: 0
  offset 116: 0
  offset 120: 9
  offset 124: 18

------------------------------------------------------------
FIELD 2
------------------------------------------------------------
Header size: 408
Structure ID: 14143

TEXT STRINGS:
